# Cross-entropy and softmax
**Machine Learning · Week 3 (Classification) · Day 2**

Lesson page: https://sara-aminpour.com/tutorials/ml/w3d2.html

Cross-entropy is the loss behind almost every classifier, and softmax turns any number of scores
into probabilities.

In [ ]:
import numpy as np
import torch

torch.set_num_threads(1)
for p in [1.0, 0.9, 0.5, 0.25, 0.01]:
    print(f"probability for the true class {p:4.2f}  ->  cross-entropy -log p = {-np.log(p) + 0:.3f}")

## Why not squared error?
The true class is 1, but the model is confidently wrong: its score is $z = -4$, so $p = \sigma(-4) \approx 0.02$.
How hard does each loss push $z$ upward? Autograd tells us.

In [ ]:
for name in ["squared error", "cross-entropy"]:
    z = torch.tensor(-4.0, requires_grad=True)
    p = torch.sigmoid(z)
    loss = (p - 1) ** 2 if name == "squared error" else -torch.log(p)
    loss.backward()
    print(f"{name:>14}: loss {loss.item():.3f}, gradient dL/dz = {z.grad.item():+.4f}")
print("Cross-entropy's gradient is simply p - y = 0.018 - 1. Squared error's is tiny: learning would crawl.")

## Softmax: scores to probabilities
$p_k = e^{z_k} / \sum_j e^{z_j}$. The **temperature** $T$ divides the scores first: small $T$ = confident, large $T$ = unsure.

In [ ]:
z = torch.tensor([2.0, 1.0, 0.1])
for T in [0.5, 1.0, 3.0]:
    p = torch.softmax(z / T, dim=0)
    print(f"T = {T}: probabilities {p.numpy().round(3)}  (sum {p.sum().item():.2f})")

## Three classes in PyTorch
`nn.Linear(2, 3)` gives one score per class (the **logits**). `nn.CrossEntropyLoss` applies the softmax and
the cross-entropy together; it wants the raw logits and the class numbers 0, 1, 2.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris

iris = load_iris()
X = torch.tensor(iris.data[:, 2:], dtype=torch.float32)       # petal length and width
X = (X - X.mean(dim=0)) / X.std(dim=0)
y = torch.tensor(iris.target)

torch.manual_seed(0)
model = torch.nn.Linear(2, 3)
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)
for step in range(300):
    optimizer.zero_grad()
    loss = loss_fn(model(X), y)
    loss.backward()
    optimizer.step()
with torch.no_grad():
    acc = (model(X).argmax(dim=1) == y).float().mean().item()
print(f"cross-entropy {loss.item():.3f}, accuracy {acc:.3f}")

xx, yy = np.meshgrid(np.linspace(-2.2, 2.2, 300), np.linspace(-2.2, 2.2, 300))
with torch.no_grad():
    grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)
    region = model(grid).argmax(dim=1).reshape(xx.shape).numpy()
colors = ["#ff8a50", "#5577f5", "#1fb394"]
plt.figure(figsize=(5.6, 4.4))
plt.contourf(xx, yy, region, levels=[-0.5, 0.5, 1.5, 2.5], colors=["#ffdcc9", "#d6defe", "#ccf2e7"])
for k in range(3):
    plt.scatter(X[y == k, 0], X[y == k, 1], s=18, color=colors[k], label=iris.target_names[k])
plt.xlabel("petal length (standardized)")
plt.ylabel("petal width (standardized)")
plt.legend()
plt.title("Softmax regression: three straight borders")
plt.show()

## Your turn: softmax by hand, safely
Write `softmax(z)` with `torch.exp` and a sum, and compare it with `torch.softmax`.
Then try the scores `[1000, 1001, 1002]`: what goes wrong, and how does subtracting the largest score fix it?

In [ ]:
def softmax(z):
    return z   # TODO: exponentiate, then divide by the sum

z = torch.tensor([2.0, 1.0, 0.1])
print("mine:   ", softmax(z))
print("PyTorch:", torch.softmax(z, dim=0))

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
def naive_softmax(z):
    e = torch.exp(z)
    return e / e.sum()

def softmax(z):
    e = torch.exp(z - z.max())          # subtracting the max changes nothing mathematically...
    return e / e.sum()

z = torch.tensor([2.0, 1.0, 0.1])
print("mine:   ", softmax(z))
print("PyTorch:", torch.softmax(z, dim=0))
big = torch.tensor([1000.0, 1001.0, 1002.0])
print("naive on big scores:", naive_softmax(big), " <- exp(1000) overflows to inf, and inf/inf = nan")
print("safe on big scores: ", softmax(big), " <- ...but keeps every exponent small")